# Most likely precipitation type — ENS map

Python/earthkit port of the Metview macro *"Plot map ... FMI case study: Finland and Estonia"*.

For each requested step the map shows:
- **Colours**: the most probable precipitation type and its probability (<50, 50–70, >70 %) where the probability of precipitation falling (PPF) is ≥ 50 %.
- **Greys**: PPF between 10–30 % and 30–50 %.

The probability of each type is the percentage of ENS members with `ptype` (param 260015) equal to that type **and**
precipitation rate (`tprate`, param 260048) above the type threshold — the same computation as the old
`ProbPrecType_calculate_all*.mv` macros that produced the `perc_<type>_<step>.grib` files read by the original map macro.

**Differences from the macro:**
- Raw ENS data are retrieved from MARS with `earthkit.data` (nearest-neighbour interpolation to a regular grid, cropped to the map domain) instead of reading pre-computed probability files.
- Requests are split by step and member chunk and run in parallel; each step is cached in `OUT_DIR` so re-runs and other step selections reuse it.
- Probabilities are accumulated member by member with NumPy (low memory) and the map is drawn with `earthkit.plots`.
- The grey legend labels are fixed: the darker grey is PPF 30–50 % and the lighter grey PPF 10–30 % (they were swapped in the macro legend).
- Since IFS 50r1 (12 May 2026) ENFO has no control forecast, so the member count is taken from the data instead of being fixed at 51.

In [ ]:
import shutil
from concurrent.futures import ThreadPoolExecutor
from math import ceil, floor
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import earthkit.data as ekd
import earthkit.plots as ekp
import numpy as np
import pandas as pd
from earthkit.plots.styles import Style, magics_colors_to_rgb
from matplotlib.patches import Patch
from scipy.ndimage import uniform_filter

## User parameters

In [ ]:
OUT_DIR = Path("/perm/moeg/Products_plotting")   # GRIB cache + plots

FDATE = "20240117"     # YYYYMMDD
TIME = "00"            # 00 / 06 / 12 / 18
EXPVER = "1"
STEPS = [3]            # e.g. list(range(0, 145, 3)) + [150, 156, 162]

# Polar stereographic map given by its lower-left and upper-right corners [lat, lon, lat, lon] (as Magics CORNERS)
MAP_CORNERS = [30, -25, 50, 65]
GRID = 0.1             # deg; ENS is interpolated (nearest neighbour) to a regular lat/lon grid

# Minimum precipitation rate (mm/h) for a member to count for each type (zero-bias calibration against SYNOP)
THRESHOLDS = {
    "rain": 0.12,
    "sleet": 0.1,
    "wetsnow": 0.05,
    "snow": 0.05,
    "icepellets": 0.05,
    "freezdrizzle": 0.02,
    "freezrain": 0.05,
}

N_PARALLEL = 8                       # concurrent MARS requests
INCLUDE_CF = FDATE < "20260512"      # IFS 50r1 removed the control forecast from ENFO

## Constants and helpers

In [ ]:
PARAM_PTYPE, PARAM_PRATE = 260015, 260048
# older cycles have no total rate (260048): use the sum of convective/large-scale rain and snowfall rates
PRATE_COMPONENTS = [228218, 228219, 228220, 228221]   # crr, lsrr, csfr, lssfr
PROJ = ccrs.NorthPolarStereo(central_longitude=0)
PLATE = ccrs.PlateCarree()


def magics_colour(c: str) -> tuple:
    if c.upper().startswith("RGB("):
        return tuple(float(v) for v in c[4:-1].split(","))
    return magics_colors_to_rgb([c.lower()])[0]


# Legend/category order; category = cat + 0/1/2 for probability <50 / 50-70 / >70 %
PTYPES = {
    "rain":         dict(code=1,  cat=1,  label="Rain",
                         colours=["RGB(0.79,0.94,0.75)", "RGB(0.23,0.89,0.08)", "RGB(0.10,0.64,0.03)"]),
    "sleet":        dict(code=7,  cat=4,  label="Sleet",
                         colours=["RGB(0.479,0.917,0.8660)", "TURQUOISE", "RGB(0.081,0.444,0.4018)"]),
    "wetsnow":      dict(code=6,  cat=7,  label="Wet snow",
                         colours=["SKY", "RGB(0.268,0.268,0.9793)", "NAVY"]),
    "snow":         dict(code=5,  cat=10, label="Snow",
                         colours=["RGB(0.829,0.757,0.9451)", "LAVENDER", "RGB(0.336,0.111,0.6970)"]),
    "icepellets":   dict(code=8,  cat=13, label="Ice pellets",
                         colours=["RGB(0.923,0.727,0.5440)", "ORANGE", "RGB(0.74,0.39,0.02)"]),
    "freezdrizzle": dict(code=12, cat=16, label="Freezing drizzle",
                         colours=["RGB(0.988,0.737,0.882)", "RGB(0.988,0.435,0.757)", "RGB(0.722,0.271,0.533)"]),
    "freezrain":    dict(code=3,  cat=19, label="Freezing rain",
                         colours=["RGB(0.96,0.51,0.54)", "RGB(1.00,0.01,0.29)", "RGB(0.75,0.02,0.30)"]),
}
PPF_CATS = {22: ("30-50", "RGB(0.64,0.64,0.64)"), 23: ("10-30", "RGB(0.83,0.83,0.83)")}
PROB_LABELS = ("<50", "50-70", ">70")
LAND_COLOUR = magics_colour("CREAM")

CAT_COLOURS = [magics_colour(c) for p in PTYPES.values() for c in p["colours"]] + \
              [magics_colour(c) for _, c in PPF_CATS.values()]
CAT_STYLE = Style(colors=CAT_COLOURS, levels=np.arange(0.5, len(CAT_COLOURS) + 1))


def map_extent(corners: list) -> tuple:
    """Projected (x0, x1, y0, y1) of the rectangle with the given lower-left/upper-right corners."""
    lat0, lon0, lat1, lon1 = corners
    (x0, y0), (x1, y1) = (PROJ.transform_point(lon, lat, PLATE) for lat, lon in ((lat0, lon0), (lat1, lon1)))
    return x0, x1, y0, y1


def retrieval_area(extent: tuple, margin: float = 1.0) -> list:
    """[N, W, S, E] lat/lon box enclosing the projected map rectangle."""
    x0, x1, y0, y1 = extent
    t = np.linspace(0, 1, 400)
    xs = np.concatenate([x0 + (x1 - x0) * t, np.full_like(t, x1), x1 - (x1 - x0) * t, np.full_like(t, x0)])
    ys = np.concatenate([np.full_like(t, y0), y0 + (y1 - y0) * t, np.full_like(t, y1), y1 - (y1 - y0) * t])
    lon, lat = PLATE.transform_points(PROJ, xs, ys)[:, :2].T
    south = max(-90, floor(lat.min() - margin))
    if x0 <= 0 <= x1 and y0 <= 0 <= y1:   # pole inside the map
        return [90, -180, south, 180]
    return [min(90, ceil(lat.max() + margin)), floor(lon.min() - margin), south, ceil(lon.max() + margin)]


def ptype_probabilities(path: Path) -> tuple[dict, int, np.ndarray, np.ndarray]:
    """% of members with each ptype and precipitation rate >= its threshold."""
    fl = ekd.from_source("file", str(path)).to_fieldlist()
    param, number = np.array(fl.get(["metadata.paramId", "metadata.number"]), dtype=int).T
    members = np.unique(number[param == PARAM_PTYPE])
    n_rate = np.unique(param[param != PARAM_PTYPE]).size

    counts = dict.fromkeys(PTYPES, 0)
    for n in members:
        pt_idx = np.flatnonzero((number == n) & (param == PARAM_PTYPE))
        pr_idx = np.flatnonzero((number == n) & (param != PARAM_PTYPE))
        if pt_idx.size != 1 or pr_idx.size != n_rate:
            raise RuntimeError(f"Incomplete fields for member {n} in {path}")
        code = np.rint(fl[int(pt_idx[0])].to_numpy())
        # total rate = sum over the rate parameter(s) in the file (260048, or its 4 components)
        rate = sum(fl[int(j)].to_numpy() for j in pr_idx) * 3600.0   # kg m-2 s-1 -> mm/h
        for name, p in PTYPES.items():
            counts[name] = counts[name] + ((code == p["code"]) & (rate >= THRESHOLDS[name]))

    n_mem = members.size
    geo = fl[0].geography
    return ({k: 100.0 * c / n_mem for k, c in counts.items()}, n_mem,
            geo.distinct_latitudes(), geo.distinct_longitudes())


def most_likely_category(perc: dict) -> np.ndarray:
    """Category 1-21 (type x probability class), 22-23 (PPF 30-50 / 10-30 %) or 0, with the macro's tie rules."""
    dry = 100.0 - sum(perc.values())
    dry = np.where(dry >= 50, dry, 0.0)
    p = {k: np.where(dry < 50, v, 0.0) for k, v in perc.items()}
    r, sl, ws, sn, ip, fd, fr = (p[k] for k in PTYPES)
    winner = {
        "rain":         (r > fr) & (r > sn) & (r > ws) & (r > ip) & (r > sl) & (r > fd),
        "sleet":        (sl > fr) & (sl > sn) & (sl > ws) & (sl > ip) & (sl > r) & (sl > fd),
        "wetsnow":      (ws > fr) & (ws > sn) & (ws >= sl) & (ws >= ip) & (ws >= r) & (ws > fd),
        "snow":         (sn > fr) & (sn >= ws) & (sn >= sl) & (sn >= ip) & (sn >= r) & (sn > fd),
        "icepellets":   (ip > fr) & (ip > sn) & (ip > sl) & (ip > ws) & (ip > r) & (ip > fd),
        "freezdrizzle": (fd > fr) & (fd >= ip) & (fd >= sn) & (fd >= sl) & (fd >= ws) & (fd >= r),
        "freezrain":    (fr >= ip) & (fr >= sn) & (fr >= sl) & (fr >= ws) & (fr >= r) & (fr > fd),
    }
    cat = np.zeros(dry.shape, dtype=np.int16)
    for name, info in PTYPES.items():
        fc = np.where(winner[name], p[name], 0.0)
        cat += np.select([(fc >= 1) & (fc < 50), (fc >= 50) & (fc <= 70), fc > 70],
                         [info["cat"], info["cat"] + 1, info["cat"] + 2], 0).astype(np.int16)
    cat += np.select([(dry > 0) & (dry < 70), (dry >= 70) & (dry <= 90)], [22, 23], 0).astype(np.int16)
    return cat


def threshold_text(thresholds: dict) -> str:
    common = pd.Series(thresholds).mode()[0]
    special = [f"{PTYPES[k]['label'].lower()}>{v:g}mm/h" for k, v in thresholds.items() if v != common]
    return "Minimum rates used are: " + ", ".join(special + [f"(all other types)>{common:g}mm/h"]) + \
        " (based on a zero bias calibration against SYNOP present weather)."

## Retrieve (or reuse cached) ENS data

In [ ]:
EXTENT = map_extent(MAP_CORNERS)
AREA = retrieval_area(EXTENT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
run_tag = f"{FDATE}{TIME}_e{EXPVER}"
area_tag = "_".join(f"{v:g}" for v in AREA)


def step_file(step: int) -> Path:
    return OUT_DIR / f"ptype_prate_map_{run_tag}_{GRID:g}deg_{area_tag}_s{step:03d}.grib"


BASE_REQUEST = {"class": "od", "stream": "enfo", "expver": EXPVER, "date": FDATE, "time": TIME,
                "levtype": "sfc", "area": AREA, "grid": [GRID, GRID], "interpolation": "nearest neighbour"}


def mars_to_file(request: dict, target: Path) -> Path:
    ekd.from_source("mars", {**BASE_REQUEST, **request}).to_target("file", str(target))
    return target


def available_prate_params(step: int) -> list[int]:
    probe = {**BASE_REQUEST, "type": "pf", "number": 1, "step": step, "param": PARAM_PRATE}
    try:
        ekd.from_source("mars", probe).to_fieldlist()
        return [PARAM_PRATE]
    except Exception:   # MARS client fails with "Expected 1, got 0" when the parameter is not archived
        return PRATE_COMPONENTS


missing = [s for s in STEPS if not step_file(s).exists()]
if missing:
    params = [PARAM_PTYPE, *available_prate_params(missing[0])]
    # few steps -> split each step into member chunks so all workers are busy
    n_chunks = max(1, N_PARALLEL // len(missing))
    parts = {}
    for s in missing:
        requests = [{"type": "pf", "number": c.tolist(), "step": s, "param": params}
                    for c in np.array_split(np.arange(1, 51), n_chunks)]
        if INCLUDE_CF:
            requests.append({"type": "cf", "step": s, "param": params})
        parts[s] = [(r, step_file(s).with_suffix(f".part{i}")) for i, r in enumerate(requests)]

    print(f"Retrieving {len(missing)} step(s), parameters {params}, with {sum(map(len, parts.values()))} MARS requests …")
    with ThreadPoolExecutor(N_PARALLEL) as pool:
        list(pool.map(lambda t: mars_to_file(*t), [t for v in parts.values() for t in v]))

    for s, step_parts in parts.items():
        # write to a temporary name first so a failed run never leaves a partial cache file
        tmp = step_file(s).with_suffix(".tmp")
        with open(tmp, "wb") as out:
            for _, part in step_parts:
                with open(part, "rb") as f:
                    shutil.copyfileobj(f, out)
                part.unlink()
        tmp.rename(step_file(s))

print("Retrieval area [N, W, S, E]:", AREA)
print("Cached steps:", [s for s in STEPS if step_file(s).exists()])

## Compute and plot the most likely precipitation type

In [ ]:
base_time = pd.Timestamp(f"{FDATE} {TIME}:00")
text1 = ("Most likely precipitation type. Greys show probability of precipitation falling (PPF) when it lies "
         "between 10 and 50%.\nColours show the most probable type, and its probability, when PPF>50%.")
text2 = threshold_text(THRESHOLDS)

# one legend column per type (header, >70, 50-70, <50) plus the PPF column
handles, labels = [], []
for p in PTYPES.values():
    handles += [Patch(visible=False)] + [Patch(facecolor=magics_colour(c), edgecolor="black", linewidth=0.5)
                                         for c in reversed(p["colours"])]
    labels += [p["label"], *reversed(PROB_LABELS)]
handles += [Patch(visible=False)] + [Patch(facecolor=magics_colour(c), edgecolor="black", linewidth=0.5)
                                     for _, c in PPF_CATS.values()] + [Patch(visible=False)]
labels += ["Precipitation <50%", *(lab for lab, _ in PPF_CATS.values()), ""]

fig_width = 11.69
fig_height = fig_width * (EXTENT[3] - EXTENT[2]) / (EXTENT[1] - EXTENT[0])   # map fills the figure

results = {}   # step -> (perc, lat, lon), reused by the hotspot search below
for step in STEPS:
    perc, n_mem, lat, lon = ptype_probabilities(step_file(step))
    results[step] = (perc, lat, lon)
    cat = most_likely_category(perc)
    valid_time = base_time + pd.Timedelta(hours=step)

    figure = ekp.Figure(rows=1, columns=1, figsize=(fig_width, fig_height))
    m = figure.add_map(crs=PROJ)
    m.ax.set_extent(EXTENT, crs=PROJ)
    m.land(color=LAND_COLOUR, zorder=0)
    m.pcolormesh(x=lon, y=lat, z=np.ma.masked_equal(cat, 0), style=CAT_STYLE, transform=PLATE)
    m.coastlines(color="black", linewidth=0.5)
    m.borders(color="black", linewidth=0.5)

    leg = m.ax.legend(handles, labels, ncol=len(PTYPES) + 1, loc="lower center", bbox_to_anchor=(0.5, 1.0),
                      frameon=False, fontsize=11, handlelength=2.2, handleheight=1.2, columnspacing=1.0,
                      handletextpad=0.5)
    for text in leg.get_texts()[::4]:
        text.set_fontweight("bold")

    # wrap=False: matplotlib's wrapped-text bbox is wrong and adds blank space to the saved figure
    figure.title(
        f"{text1}\n{text2}\n"
        f"ENS forecast ({n_mem} members) from: {base_time:%A, %d-%m-%Y %H} UTC          "
        f"VT: T+{step} = {valid_time:%A, %d-%m-%Y %H} UTC",
        fontsize=10, y=1.12, wrap=False,
    )
    figure.add_logo("ecmwf")

    out_png = OUT_DIR / f"Prob_prec_type_map{FDATE}_{TIME}_{step:03d}.png"
    figure.save(out_png, dpi=150)
    print("Saved", out_png)
    figure.show()

## Find locations with the highest probability of a precipitation type

Ranks the grid points inside the map by the probability of `TARGET_TYPE` at `TARGET_STEP` (ties broken by the
mean probability in a ~0.5° neighbourhood), keeps points at least `MIN_SEPARATION_KM` apart, and names the nearest
populated place (Natural Earth). Copy `lat`/`lon` of the chosen row into the meteogram notebook.

In [ ]:
TARGET_TYPE = "freezrain"
TARGET_STEP = STEPS[0]
TOP_N = 5
MIN_SEPARATION_KM = 100


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


perc, lat, lon = results[TARGET_STEP]
lon2d, lat2d = np.meshgrid(lon, lat)
xy = PROJ.transform_points(PLATE, lon2d, lat2d)
x0, x1, y0, y1 = EXTENT
inside = (xy[..., 0] >= x0) & (xy[..., 0] <= x1) & (xy[..., 1] >= y0) & (xy[..., 1] <= y1)
prob = np.where(inside, perc[TARGET_TYPE], 0.0)
smooth = uniform_filter(prob, size=max(1, round(0.5 / GRID)), mode="nearest")

places = list(shpreader.Reader(shpreader.natural_earth("10m", "cultural", "populated_places")).records())
place_lat = np.array([p.attributes["LATITUDE"] for p in places])
place_lon = np.array([p.attributes["LONGITUDE"] for p in places])

picks = []
for k in np.lexsort((smooth.ravel(), prob.ravel()))[::-1]:
    if prob.flat[k] <= 0 or len(picks) == TOP_N:
        break
    la, lo = lat2d.flat[k], lon2d.flat[k]
    if any(haversine_km(la, lo, p["lat"], p["lon"]) < MIN_SEPARATION_KM for p in picks):
        continue
    dist = haversine_km(la, lo, place_lat, place_lon)
    j = int(np.argmin(dist))
    picks.append({"lat": round(float(la), 2), "lon": round(float(lo), 2),
                  f"prob_{TARGET_TYPE}_%": round(float(prob.flat[k]), 1),
                  "neighbourhood_mean_%": round(float(smooth.flat[k]), 1),
                  "nearest_place": places[j].attributes["NAME"],
                  "country": places[j].attributes["ADM0NAME"],
                  "distance_km": round(float(dist[j]), 1)})

hotspots = pd.DataFrame(picks)
print(f"{PTYPES[TARGET_TYPE]['label']} at T+{TARGET_STEP} "
      f"(valid {base_time + pd.Timedelta(hours=TARGET_STEP):%Y-%m-%d %H} UTC)")
hotspots